## Prandtl-meyer expansion

https://www.grc.nasa.gov/www/wind/valid/pm15/pm15.html

In [ ]:
# main

// pm
int main(){
    Mesh mesh;
    mesh.load("mesh/pm_nasa_v1");

    Solver solver(mesh);
    solver.setNumericalMethod(FluxScheme::ROE, SpatialScheme::SECOND_ORDER_CENTRAL);
    solver.initialize();

    std::cout<< "Start implicit" << std::endl;

    solver.solveImplicit(1e-6, 500000);
    std::cout << "Done implicit" << std::endl;
    saveResults(mesh, solver.getSolution(), 9999);
    solver.extractWallPressure("results/pm_centerline.csv");
    std::cout<< "done" << std::endl;
    return 0;
}

In [ ]:
# initialize


void Solver::initialize() {
    this->gamma = 1.4;
    this->R_gas = 1716.48;  

    this->M_inf = 2.5;
    
    double P0_psia = 12.0;    
    double T0_R = 550.0;   

    double factor_T = 1.0 + 0.5 * (this->gamma - 1.0) * this->M_inf * this->M_inf;
    double p_psia = P0_psia / std::pow(factor_T, this->gamma / (this->gamma - 1.0));
    double T = T0_R / factor_T;

    double p = p_psia * 144.0;
    
    double c = std::sqrt(this->gamma * this->R_gas * T); 
    double u = this->M_inf * c; 
    double v = 0.0;
    double rho = p / (this->R_gas * T); 
    
    State init = primitiveToConservative(rho, u, v, p);
    std::fill(U_current.begin(), U_current.end(), init);
    
    std::cout << "Field initialized: PM Expansion (P_static_psia = " << p_psia << ")" << std::endl;
}

In [ ]:
# ghost state

In [ ]:
State Solver::getBoundaryGhostState(const State& U_in, const Face& f) {
    State U_ghost = U_in;

    // --- BC Tag 1: Supersonic Inlet (Frozen Dirichlet) ---
    if (f.bcTag == 1) {
        this->R_gas = 1716.48;     
        double P0_psia = 12.0;   
        double T0_R = 550.0;
        double M = 2.5;
        
        double factor_T = 1.0 + 0.5 * (gamma - 1.0) * M * M;
        double p_psia = P0_psia / std::pow(factor_T, gamma / (gamma - 1.0));
        double T = T0_R / factor_T;
        
        double p = p_psia * 144.0;
        
        double c = std::sqrt(gamma * R_gas * T);
        double rho = p / (R_gas * T);
        double u = M * c;
        double v = 0.0;
        
        U_ghost = primitiveToConservative(rho, u, v, p);
    }
    // --- BC Tag 2: Supersonic Outlet (Extrapolation) ---
    else if (f.bcTag == 2) {
        // Zero-gradient for supersonic outflow: ghost state perfectly mirrors inside
        U_ghost = U_in; 
    }
    // --- BC Tag 3: Inviscid Wall (Slip Wall) ---
    else if (f.bcTag == 3) {
        double u = U_in.rhou / U_in.rho;
        double v = U_in.rhov / U_in.rho;
        
        // Wall normal velocity
        double vn = u * f.nx + v * f.ny;
        
        // Reflect velocity across the face normal
        double u_g = u - 2.0 * vn * f.nx;
        double v_g = v - 2.0 * vn * f.ny;
        
        U_ghost.rho = U_in.rho;
        U_ghost.rhou = U_in.rho * u_g;
        U_ghost.rhov = U_in.rho * v_g;
        U_ghost.rhoE = U_in.rhoE; 
    }

    return U_ghost;
}

In [ ]:
# compute residual

TotalFlux = FluxEuler - FluxVisc;    ---- >   TotalFlux = FluxEuler;

## SWBLI

https://www.grc.nasa.gov/WWW/wind/valid/m5swbli/m5swbli.html

In [ ]:
# main 


// SWBLI mach 5 shock wave:
int main() {
    Mesh mesh;
    mesh.load("mesh/swbli_8x"); 

    Solver solver(mesh);
    solver.setFreestreamStatic(5.0, 4006.88, 68.33);
   
    solver.setKinematicViscosity(1.88e7, 0.5);
    solver.initialize();
    
    std::cout<< "First part of JFNK" << std::endl;
    solver.setNumericalMethod(FluxScheme::HLLC, SpatialScheme::FIRST_ORDER);
    solver.solveImplicit(1e3, 100000);

    std::cout << "Starting NASA Mach 5 SWBLI simulation..." << std::endl;
    solver.setNumericalMethod(FluxScheme::HLLC, SpatialScheme::SECOND_ORDER_LIMITED);
    solver.solveImplicit(1e-4, 200000);
    solver.extractSkinFriction("swbli_cf_profile.csv");
    saveResults(mesh, solver.getSolution(), 9999);
    return 0;   
}


In [ ]:
# initialize

void Solver::initialize() {
    // calculate T_static and p_static from total conditions and Mach number
    double factor_T = 1.0 + 0.5 * (gamma - 1.0) * M_inf * M_inf;
    double T_static = T_total_inlet / factor_T;
    double p_static = P_static_exit; 
    
    double rho = p_static / (R_gas * T_static);
    double c = std::sqrt(gamma * R_gas * T_static);
    double u = M_inf * c; // zero angle
    double v = 0.0;
    
    State init = primitiveToConservative(rho, u, v, p_static);
    std::fill(U_current.begin(), U_current.end(), init);
    
    std::cout << "Field initialized with Ma = " << M_inf << ", P = " << p_static << std::endl;
}

In [ ]:
# ghostboundary state
// for SWBLI mach 5 case
State Solver::getBoundaryGhostState(const State& U_in, const Face& f) {
    State U_ghost = U_in;

    // --- BC Tag 1 & 5: Subsonic Inlet & Farfield ---
    if (f.bcTag == 1 || f.bcTag == 5) {
        // Enforce incoming Mach, T, and P. 
        double factor_T = 1.0 + 0.5 * (gamma - 1.0) * M_inf * M_inf;
        double T_static = T_total_inlet / factor_T;
        double rho_inf = P_static_exit / (R_gas * T_static);
        double c_inf = std::sqrt(gamma * R_gas * T_static);
        double u_inf = M_inf * c_inf;
        
        U_ghost = primitiveToConservative(rho_inf, u_inf, 0.0, P_static_exit);
    }
    // --- BC Tag 2: Subsonic Outlet ---
    else if (f.bcTag == 2) {
        // for mach 5 case:
        U_ghost = U_in;
    }
    // --- BC Tag 3: Slip Wall (Upstream of leading edge) ---
    else if (f.bcTag == 3) {
        double u = U_in.rhou / U_in.rho;
        double v = U_in.rhov / U_in.rho;
        double vn = u * f.nx + v * f.ny;
        
        double u_g = u - 2.0 * vn * f.nx;
        double v_g = v - 2.0 * vn * f.ny;
        
        U_ghost.rho = U_in.rho;
        U_ghost.rhou = U_in.rho * u_g;
        U_ghost.rhov = U_in.rho * v_g;
        U_ghost.rhoE = U_in.rhoE; 
    }
    // --- BC Tag 4: Flat Plate (No-Slip Wall) ---
    else if (f.bcTag == 4) {
        // for mach 5 case:
        double u_g = - (U_in.rhou / U_in.rho);
        double v_g = - (U_in.rhov / U_in.rho);

        double ek_in = 0.5 * (U_in.rhou * U_in.rhou + U_in.rhov * U_in.rhov) / U_in.rho;
        double p_in = (gamma - 1.0) * (U_in.rhoE - ek_in);
        double T_in = p_in / (U_in.rho * R_gas);

        double T_ghost = T_in;
        double rho_g = p_in / (R_gas * T_ghost);
        double k_in = 0.0, omega_in = 0.0;

        U_ghost = primitiveToConservative(rho_g, u_g, v_g, p_in, k_in, omega_in);
    }

    return U_ghost;
}


## laminar flat plate
https://www.grc.nasa.gov/www/wind/valid/fplam/fplam01/fplam01.html

In [ ]:
# main

int main() {
    Mesh mesh;
    // Load your flat plate unstructured mesh
    mesh.load("mesh/new_laminar_flat_plate"); 

    Solver solver(mesh);

    solver.setNumericalMethod(FluxScheme::HLLC, SpatialScheme::SECOND_ORDER_CENTRAL);
    // Convert Flow conditions to SI if needed: 
    // Mach 0.1, P_static = 41368.5 Pa, T_static = 388.89 K
    solver.setFlowConditions(0.1, 41368.5, 388.89);
    
    // Add Viscosity initialization for flat plate (Re=200k, L=0.3048m)
    solver.setKinematicViscosity(200000.0, 0.3048);
    
    solver.initialize();

    std::cout << "Starting simulation JFNK..." << std::endl;
    // You might need a smaller CFL or local timestep since viscous cells near the wall are tiny
    solver.solveImplicit(1e-4, 50000); 
    double L = 0.3048;
    solver.extractBoundaryLayerProfile(L, "bl_profile.csv");
    solver.extractSkinFriction("cf_profile.csv");
    saveResults(mesh, solver.getSolution(), 9999);
    return 0;
}


In [ ]:
# initialize
void Solver::initialize() {
    // calculate T_static and p_static from total conditions and Mach number
    double factor_T = 1.0 + 0.5 * (gamma - 1.0) * M_inf * M_inf;
    double T_static = T_total_inlet / factor_T;
    double p_static = P_static_exit; 
    
    double rho = p_static / (R_gas * T_static);
    double c = std::sqrt(gamma * R_gas * T_static);
    double u = M_inf * c; // zero angle
    double v = 0.0;
    
    State init = primitiveToConservative(rho, u, v, p_static);
    std::fill(U_current.begin(), U_current.end(), init);
    
    std::cout << "Field initialized with Ma = " << M_inf << ", P = " << p_static << std::endl;
}


In [ ]:
# ghost state

State Solver::getBoundaryGhostState(const State& U_in, const Face& f) {
    State U_ghost = U_in;

    // --- BC Tag 1 & 5: Subsonic Inlet & Farfield ---
    if (f.bcTag == 1 || f.bcTag == 5) {
        // Enforce incoming Mach, T, and P. 
        double factor_T = 1.0 + 0.5 * (gamma - 1.0) * M_inf * M_inf;
        double T_static = T_total_inlet / factor_T;
        double rho_inf = P_static_exit / (R_gas * T_static);
        double c_inf = std::sqrt(gamma * R_gas * T_static);
        double u_inf = M_inf * c_inf;
        
        U_ghost = primitiveToConservative(rho_inf, u_inf, 0.0, P_static_exit);
    }
    // --- BC Tag 2: Subsonic Outlet ---
    else if (f.bcTag == 2) {
        // Enforce fixed pressure, extrapolate velocity and density
        // double rho = U_in.rho;
        // double u = U_in.rhou / U_in.rho;
        // double v = U_in.rhov / U_in.rho;
        // U_ghost = primitiveToConservative(rho, u, v, P_static_exit);


        // for mach 5 case:
        U_ghost = U_in;
    }
    // --- BC Tag 3: Slip Wall (Upstream of leading edge) ---
    else if (f.bcTag == 3) {
        double u = U_in.rhou / U_in.rho;
        double v = U_in.rhov / U_in.rho;
        double vn = u * f.nx + v * f.ny;
        
        double u_g = u - 2.0 * vn * f.nx;
        double v_g = v - 2.0 * vn * f.ny;
        
        U_ghost.rho = U_in.rho;
        U_ghost.rhou = U_in.rho * u_g;
        U_ghost.rhov = U_in.rho * v_g;
        U_ghost.rhoE = U_in.rhoE; 
    }
    // --- BC Tag 4: Flat Plate (No-Slip Wall) ---
    else if (f.bcTag == 4) {
        // Reverse velocity to enforce zero flow
        // double u_g = - (U_in.rhou / U_in.rho);
        // double v_g = - (U_in.rhov / U_in.rho);
        
        // U_ghost.rho = U_in.rho;
        // U_ghost.rhou = U_in.rho * u_g;
        // U_ghost.rhov = U_in.rho * v_g;
        // U_ghost.rhoE = U_in.rhoE; 

        // for mach 5 case:
        double u_g = - (U_in.rhou / U_in.rho);
        double v_g = - (U_in.rhov / U_in.rho);

        double ek_in = 0.5 * (U_in.rhou * U_in.rhou + U_in.rhov * U_in.rhov) / U_in.rho;
        double p_in = (gamma - 1.0) * (U_in.rhoE - ek_in);
        double T_in = p_in / (U_in.rho * R_gas);

        // double T_wall = 300.0;
        // double T_ghost = 2.0 * T_wall - T_in; // mirror temperature for adiabatic wall
        // if (T_ghost <= 10.0) T_ghost = T_wall;
        double T_ghost = T_in;
        double rho_g = p_in / (R_gas * T_ghost);
        double k_in = 0.0, omega_in = 0.0;

        U_ghost = primitiveToConservative(rho_g, u_g, v_g, p_in, k_in, omega_in);
    }

    return U_ghost;
}

## Wind tunnel

In [ ]:
# main

// for wind tunnel

int main() {
    Mesh mesh;
    // read mesh
    mesh.load("mesh/wind_tunnel_4t"); 

    Solver solver(mesh);

    // 2. initialize
    solver.initialize();

    // 3. explicit time stepping
    
    // double dt = 5e-4;
    // int maxSteps = 20000; 

    // std::cout << "Starting Explicit Mach 3 Simulation..." << std::endl;
    // for (int i = 0; i <= maxSteps; ++i) {
    //     solver.timeStepExplicit(dt);
      
        
    //     if (i % 100 == 0) {
    //         std::cout << "Step " << i << " | Physical Time: " << i * dt << std::endl;
    //         saveResults(mesh, solver.getSolution(), i);
    //     }
    // }
    std::cout<<"Start JFNK" << std::endl;
    solver.setNumericalMethod(FluxScheme::LLF, SpatialScheme::FIRST_ORDER);
    solver.solveImplicit(1e-5, 5000);
    saveResults(mesh, solver.getSolution(), 9999);

    std::cout << "Done" << std::endl;

    return 0;
}


In [ ]:
# initialize

void Solver::initialize(){
    double rho = 1.4;
    double u = 3.0;
    double v = 0.0;
    double p = 1.0;

    State init = primitiveToConservative(rho, u, v, p);
    std::fill(U_current.begin(), U_current.end(), init);

    std::cout << "Field initialized with Mach 3 forward step condition." << std::endl;
}

In [ ]:
# boundarystate

State Solver::getBoundaryGhostState(const State& U_in, const Face& f) {
    State U_ghost = U_in;

    // --- BC Tag 1: Supersonic Inlet ---
    if (f.bcTag == 1) {
        double rho_ghost = 1.4;
        double u_ghost   = 3.0;
        double v_ghost   = 0.0;
        double p_ghost   = 1.0;
        U_ghost = primitiveToConservative(rho_ghost, u_ghost, v_ghost, p_ghost);
    }
    // --- BC Tag 2: Supersonic Outlet ---
    else if (f.bcTag == 2) {
        
        U_ghost = U_in; 
    }
    // --- BC Tag 3: Slip Wall ---
    else if (f.bcTag == 3) {
        double u = U_in.rhou / U_in.rho;
        double v = U_in.rhov / U_in.rho;
        
        double vn = u * f.nx + v * f.ny;
        
        
        double u_g = u - 2.0 * vn * f.nx;
        double v_g = v - 2.0 * vn * f.ny;
        
        U_ghost.rho = U_in.rho;
        U_ghost.rhou = U_in.rho * u_g;
        U_ghost.rhov = U_in.rho * v_g;
        U_ghost.rhoE = U_in.rhoE; 
    }

    return U_ghost;
}

## Bump

In [ ]:
# main

int main() {
    Mesh mesh;
    // mesh
    mesh.load("mesh/bump_1230"); 

    Solver solver(mesh);

    // Ma = 0.5, AoA = 0
    // P_static = 101325 Pa, T_static = 300 K

    solver.setFlowConditions(0.5, 101325.0, 300.0);
    
    solver.initialize();

    // explicit time stepping
    // L and c for calculate CFL number (explicit)
    // dt ~ L / c ~ 0.01 / 340 ~ 3e-5
    
    // double dt = 1e-5; 
    // int maxSteps = 5000;

    // std::cout << "Starting simulation..." << std::endl;
    // for (int i = 0; i <= maxSteps; ++i) {
    //     solver.timeStepExplicit(dt);
      
    //     if (i % 100 == 0) {
    //         std::cout << "Step " << i << std::endl;
    //         saveResults(mesh, solver.getSolution(), i);
    //     }
    // }

    std::cout << "JFNK" << std::endl;
    solver.setNumericalMethod(FluxScheme::LLF, SpatialScheme::FIRST_ORDER);
    solver.solveImplicit(1e-5, 2500);
    saveResults(mesh, solver.getSolution(), 9999);

    std::cout << " Grs " << std::endl;
    solver.calculateEntropyError();

    std::cout << " Drag " << std::endl;
    solver.calculateDrag();

    std::cout << "Done" << std::endl;



    return 0;
    }

In [ ]:
# initialize

// for 2d bump
void Solver::initialize() {
    // inititalize : uniform flow
    
    double p = P_static_exit;
    // T = T0 / (1 + (g-1)/2 * M^2) 
    double factor_T = std::pow(P_total_inlet / p, (gamma - 1.0) / gamma); // Isentropic relation
    double T = T_total_inlet / factor_T;
    
    double rho = p / (R_gas * T);
    double c = std::sqrt(gamma * R_gas * T);
    
    // u^2 = M^2 * c^2 
    double V = std::sqrt(2.0 * (gamma * R_gas / (gamma - 1.0)) * (T_total_inlet - T));
    
    
    State init = primitiveToConservative(rho, V, 0.0, p);
    std::fill(U_current.begin(), U_current.end(), init);
    
    std::cout << "Field initialized with M=" << V/c << ", P=" << p << std::endl;
}


In [ ]:
# boundary ghost state


// for 2d bump
// // set boundary ghost state
State Solver::getBoundaryGhostState(const State& U_in, const Face& f) {
    State U_ghost = U_in;

    // --- BC Tag 1: Subsonic Inlet (Given P0, T0) ---
    if (f.bcTag == 1) {
        
        
        double p_in = getPressure(U_in);
        
        // avoid NaN value
        double p_use = std::min(p_in, P_total_inlet - 1e-4); 

        //Isentropic relations:
        // T = T0 * (P/P0)^((g-1)/g)
        double T_ghost = T_total_inlet * std::pow(p_use / P_total_inlet, (gamma - 1.0) / gamma);
        double rho_ghost = p_use / (R_gas * T_ghost);
        
        // Velocity magnitude from Energy equation: V^2 = 2Cp(T0 - T)
        double Cp = gamma * R_gas / (gamma - 1.0);
        double V2 = 2.0 * Cp * (T_total_inlet - T_ghost);
        double V = (V2 > 0) ? std::sqrt(V2) : 0.0;

       
        U_ghost = primitiveToConservative(rho_ghost, V, 0.0, p_use);
    }
    // --- BC Tag 2: Outlet (Given Static Pressure) ---
    else if (f.bcTag == 2) {
        // subsonic outlet P_exit
        
        double p_target = P_static_exit;
        
        double rho = U_in.rho;
        double u = U_in.rhou / U_in.rho;
        double v = U_in.rhov / U_in.rho;
        
        // set others
        U_ghost = primitiveToConservative(rho, u, v, p_target);
    }
    // --- BC Tag 3: Slip Wall ---
    else if (f.bcTag == 3) {
        
        double u = U_in.rhou / U_in.rho;
        double v = U_in.rhov / U_in.rho;
        
        double vn = u * f.nx + v * f.ny;
        
        // V_ghost
        double u_g = u - 2.0 * vn * f.nx;
        double v_g = v - 2.0 * vn * f.ny;
        
        U_ghost.rho = U_in.rho;
        U_ghost.rhou = U_in.rho * u_g;
        U_ghost.rhov = U_in.rho * v_g;
        U_ghost.rhoE = U_in.rhoE; 
    }

    return U_ghost;
}